In [42]:
import zarr
import dask.array as da
import pandas as pd
import numpy as np

In [2]:
df = pd.read_pickle('/Volumes/OPERA3/Nathan/data/macrohet/macrohet_results/dfs/sc_df.pkl')

In [3]:
df.keys()

Index(['Time (hours)', 'Mtb Area (µm)', 'dMtb Area (µm)', 'Mphi Area (µm)',
       'dMphi Area (µm)', 'Infection Status', 'Initial Infection Status',
       'Final Infection Status', 'x', 'y', 'GFP', 'RFP', 'Eccentricity', 'MSD',
       'Technical Replicate', 'Biological Replicate', 'Strain', 'Compound',
       'Concentration', 'Cell ID', 'Acquisition ID', 'Experiment ID',
       'Unique ID', 'ID', 'Edge Status', 'Uptake',
       'dMtb Area between frames (µm)', 'Mtb Area Processed (µm)',
       'Time Model (hours)', 'Mtb Area Model (µm)', 'mtb_origin',
       'Doubling Amounts', 'Doubling Times', 'r2', 'Frame', 'category_rank'],
      dtype='object')

In [11]:
subset_df = df[df['mtb_origin']=='Growth']
subset_df

,Time (hours),Mtb Area (µm),dMtb Area (µm),Mphi Area (µm),dMphi Area (µm),Infection Status,Initial Infection Status,Final Infection Status,x,y,...,dMtb Area between frames (µm),Mtb Area Processed (µm),Time Model (hours),Mtb Area Model (µm),mtb_origin,Doubling Amounts,Doubling Times,r2,Frame,category_rank
405,0.0,46.797680,136.772588,660.776979,-68.386294,NaN,1.0,1.0,519.922607,876.779602,...,NaN,NaN,NaN,NaN,Growth,"[50.6, 101.2]",[28.0],0.97,0,NaN
406,1.0,48.719647,136.772588,585.105086,-68.386294,NaN,1.0,1.0,522.290833,876.766357,...,1.921968,NaN,NaN,NaN,Growth,"[50.6, 101.2]",[28.0],0.97,1,NaN
407,2.0,52.206007,136.772588,582.020998,-68.386294,True,1.0,1.0,524.336243,874.563110,...,3.486360,NaN,2.0,47.625714,Growth,"[50.6, 101.2]",[28.0],0.97,2,NaN
408,3.0,50.552221,136.772588,572.232372,-68.386294,True,1.0,1.0,516.952454,876.656799,...,-1.653786,NaN,3.0,49.099089,Growth,"[50.6, 101.2]",[28.0],0.97,3,NaN
409,4.0,54.463202,136.772588,590.669853,-68.386294,True,1.0,1.0,521.947449,880.909363,...,3.910981,50.552221,4.0,50.593160,Growth,"[50.6, 101.2]",[28.0],0.97,4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1906240,67.0,69.324929,NaN,1343.164920,NaN,NaN,0.0,1.0,1186.000000,1359.000000,...,NaN,6.704539,NaN,NaN,Growth,"[1.92, 3.84]",[5.0],0.85,134,NaN
1906241,67.5,73.146516,NaN,1805.621646,NaN,NaN,0.0,1.0,1158.000000,1359.000000,...,NaN,25.600163,NaN,NaN,Growth,"[1.92, 3.84]",[5.0],0.85,135,NaN
1906242,68.0,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1228.000000,1372.000000,...,NaN,40.366910,NaN,NaN,Growth,"[1.92, 3.84]",[5.0],0.85,136,NaN
1906243,68.5,0.000000,NaN,361.374632,NaN,NaN,0.0,1.0,1235.000000,1368.000000,...,NaN,40.366910,NaN,NaN,Growth,"[1.92, 3.84]",[5.0],0.85,137,NaN


In [12]:
n_tracks = subset_df.groupby(['Experiment ID', 'Acquisition ID']).size()

In [13]:
n_tracks.nlargest(1)

Experiment ID  Acquisition ID
PS0000         (4, 5)            3883
dtype: int64

In [18]:
image_fn = '/Volumes/OPERA3/Nathan/data/macrohet/PS0000/acquisition/zarr/(4, 5).zarr'
images = da.from_zarr(f"{image_fn}/images").max(axis=2)

In [19]:
images

dask.array<max-aggregate, shape=(75, 2, 6048, 6048), dtype=uint16, chunksize=(1, 1, 6048, 6048), chunktype=numpy.ndarray>

In [43]:
# Selects rows where 'Experiment ID' equals 'PS0000' AND 'Acquisition ID' equals '(4, 5)'.
tracks = df[(df['Experiment ID'] == 'PS0000') & (df['Acquisition ID'] == (4, 5) )][['Cell ID','Frame','y','x']].dropna().to_numpy(dtype=np.float64)
tracks

array([[1006.        ,    4.        , 1189.13110352,  926.14569092],
       [1006.        ,    5.        , 1176.62390137,  931.15734863],
       [1006.        ,    6.        , 1190.92285156,  946.45581055],
       ...,
       [ 996.        ,   72.        ,  684.67504883,  639.98370361],
       [ 996.        ,   73.        ,  688.83764648,  635.66717529],
       [ 996.        ,   74.        ,  685.81958008,  635.29003906]])

In [30]:
import napari

In [46]:
# viewer = napari.Viewer(title='hpig animation dev')
# viewer.add_image(images[0], channel_axis=0, colormap=['green','magenta'])
viewer.add_tracks(tracks,scale=(1,5.04,5.04))

<Tracks layer 'tracks [1]' at 0x37f6942e0>

In [47]:
%%time
images[1].compute()

CPU times: user 617 ms, sys: 516 ms, total: 1.13 s
Wall time: 14min 2s


array([[[  0,   0,   0, ...,   0,   0,   0],
        [140, 146, 135, ..., 407, 351,   0],
        [139, 146, 147, ..., 406, 366,   0],
        ...,
        [142, 142, 131, ..., 110, 108,   0],
        [135, 147, 142, ..., 112, 102,   0],
        [143, 147, 147, ..., 107, 115,   0]],

       [[  0,   0,   0, ...,   0,   0,   0],
        [104, 103, 106, ..., 107, 110,   0],
        [109, 109, 109, ..., 112, 105,   0],
        ...,
        [111, 116, 115, ..., 107, 109,   0],
        [109, 115, 109, ..., 106, 106,   0],
        [115, 116, 104, ..., 107, 109,   0]]], dtype=uint16)

In [49]:
images[0:2]

dask.array<getitem, shape=(2, 2, 6048, 6048), dtype=uint16, chunksize=(1, 1, 6048, 6048), chunktype=numpy.ndarray>

In [50]:
test = images[0:2].compute()

In [51]:
test_tracks = df[(df['Experiment ID'] == 'PS0000') & (df['Acquisition ID'] == (4, 5) ) & (df['Frame'] <= 1 )][['Cell ID','Frame','y','x']].dropna().to_numpy(dtype=np.float64)
test_tracks

array([[1.04000000e+02, 0.00000000e+00, 7.09874268e+02, 4.74159943e+02],
       [1.04000000e+02, 1.00000000e+00, 7.09386047e+02, 4.80032410e+02],
       [1.06000000e+02, 0.00000000e+00, 6.54075073e+02, 1.08217041e+03],
       ...,
       [9.60000000e+01, 1.00000000e+00, 7.00252625e+02, 2.80450470e+02],
       [9.90000000e+01, 0.00000000e+00, 6.96347839e+02, 3.72126373e+02],
       [9.90000000e+01, 1.00000000e+00, 6.96347839e+02, 3.72126373e+02]])

In [69]:
images = test

In [53]:
# viewer_2 = napari.Viewer(title = 'testing multi frame')
viewer_2.add_image(test, channel_axis=1, colormap=['green','magenta'])
# viewer_2.add_tracks(test_tracks,scale=(1,5.04,5.04))

[<Image layer 'Image' at 0x3edaeae90>,
 <Image layer 'Image [1]' at 0x3317af940>]

/Users/dayn/miniforge3/envs/godspee/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


In [57]:
layer.name

'test_tracks'

In [61]:
6048/75

80.64

In [62]:
for layer in viewer_2.layers:
    
    if layer.name == 'test_tracks':
        layer.scale = (80, 5.04, 5.04)
    else:
        layer.scale = (80, 1, 1)

/Users/dayn/miniforge3/envs/godspee/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
/Users/dayn/miniforge3/envs/godspee/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


In [67]:
images_loaded = images

In [68]:
images_loaded

dask.array<max-aggregate, shape=(75, 2, 6048, 6048), dtype=uint16, chunksize=(1, 1, 6048, 6048), chunktype=numpy.ndarray>

In [116]:
images.shape

(2, 2, 6048, 6048)

In [142]:
v = napari.Viewer(title = 'actual execution of animation')

for frame_n in range(len(images)):
    for i in reversed(range(len(v.layers))):
        v.layers.pop(i) 
    v.add_image(images[0:frame_n], channel_axis=0, colormap=['green','magenta'])
    if frame_n==0:
        continue
    current_tracks = df[(df['Experiment ID'] == 'PS0000') 
                      & (df['Acquisition ID'] == (4, 5) ) 
                      & (df['Frame'] <= frame_n )][['Cell ID','Frame','y','x']].dropna().to_numpy(dtype=np.float64)
    v.add_tracks(current_tracks, scale = (80, 5.04, 5.04))
    v.dims.ndisplay = 3
    v.camera.center = view_dictionary['center']
    v.camera.zoom = view_dictionary['zoom']
    v.camera.angles = view_dictionary['angles']
    v.camera.perspective = view_dictionary['perspective']
    if frame_n==2:
        break

/Users/dayn/miniforge3/envs/godspee/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (2, 6048, 6048) exceeds GL_MAX_TEXTURE_SIZE 2048 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(


In [129]:
v.camera.center

(39.75, 3023.5, 3023.5)

In [ ]:
v.camera = 

In [133]:
view_dictionary={}
for element in v.camera:
    print(element)
    view_dictionary[element[0]] = element[1]
    # v.camera.center = v.camera.center

('center', (39.75, 3023.5, 3023.5))
('zoom', 0.139484126984127)
('angles', (30.987817719483903, 35.572853672744216, 48.907463346476064))
('perspective', 0.0)
('mouse_pan', True)
('mouse_zoom', True)
('orientation', (<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>))


In [140]:
v.camera.center = view_dictionary['center']
v.camera.zoom = view_dictionary['zoom']
v.camera.angles = view_dictionary['angles']
v.camera.perspective = view_dictionary['perspective']


In [134]:
view_dictionary

{'center': (39.75, 3023.5, 3023.5),
 'zoom': 0.139484126984127,
 'angles': (30.987817719483903, 35.572853672744216, 48.907463346476064),
 'perspective': 0.0,
 'mouse_pan': True,
 'mouse_zoom': True,
 'orientation': (<DepthAxisOrientation.TOWARDS: 'towards'>,
  <VerticalAxisOrientation.DOWN: 'down'>,
  <HorizontalAxisOrientation.RIGHT: 'right'>)}